# Clustering des Heart-Disease-Datensatzes

In diesem Notebook untersuchen wir den UCI Heart-Disease-Datensatz mit k-Means:

1. Clusteranzahl mit der Elbow-Methode bestimmen
2. Clustering mit dem gewaehlten k durchfuehren
3. Daten mit t-SNE in 2D projizieren
4. Projizierte Daten samt Clustern im Scatterplot darstellen

## 1. Daten einlesen und aufbereiten

Wir verwenden nur die numerischen Spalten und skalieren sie, da k-Means und t-SNE mit Abständen arbeiten.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE

In [ ]:
# Datensatz einlesen
data = pd.read_csv('../Datensätze/heart_disease_uci.csv')
data.head()

In [ ]:
# Nur numerische Spalten verwenden und fehlende Werte entfernen
#X = data[['age', "trestbps", "chol", "thalch", "oldpeak", "ca", "num"]].dropna()
X = data[['age', 'chol', 'num' ]].dropna()

# Skalieren (wichtig fuer abstandsbasierte Verfahren)
scaler = StandardScaler()
X_transformed = scaler.fit_transform(X)
print('Form der Daten:', X_transformed.shape)

## 2. Clusteranzahl mit der Elbow-Methode bestimmen

Wir fuehren k-Means fuer verschiedene k aus und tragen die Zielfunktion (`inertia_`, die Summe der quadrierten Abstaende) gegen k auf. Der Knick in der Kurve zeigt ein sinnvolles k an.

In [ ]:
scores = []
ks = range(2, 10)
for n in ks:
    model = KMeans(n_clusters=n, random_state=0, n_init=10)
    model.fit(X_transformed)
    scores.append(model.inertia_)

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(list(ks), scores, marker='o')
plt.xlabel('Anzahl Cluster k')
plt.ylabel('Inertia (WCSS)')
plt.title('Elbow-Methode')
plt.tight_layout()
plt.show()

Anhand des (leichten) Knicks wählen wir eine passende Clusterzahl. Im Folgenden verwenden wir `k = 4`

## 3. Clustering mit dem gewählten k

Wir clustern die skalierten Daten mit dem gewählten k und fragen die Cluster-Labels ab.

In [ ]:
k = 4
kmeans = KMeans(n_clusters=k, random_state=0)
kmeans.fit(X_transformed)

labels = kmeans.labels_
print('Gefundene Cluster:', np.unique(labels))

## 4. Projektion mit t-SNE und Scatterplot

Die hochdimensionalen Daten werden mit t-SNE in 2D projiziert. Anschliessend färben wir die projizierten Punkte nach ihrem k-Means-Cluster ein.

In [ ]:
tsne = TSNE(n_components=2, random_state=0)
X_2d = tsne.fit_transform(X_transformed)
print('Form nach t-SNE:', X_2d.shape)

In [ ]:
plt.figure(figsize=(7, 5))
scatter = plt.scatter(X_2d[:, 0], X_2d[:, 1],
                      c=labels, cmap='viridis', s=30)
plt.xlabel('t-SNE 1')
plt.ylabel('t-SNE 2')
plt.title(f'Heart-Disease-Cluster (k = {k}), t-SNE-Projektion')
plt.legend(*scatter.legend_elements(), title='Cluster')
plt.tight_layout()

plt.show()

## Hinweise

- Die **Elbow-Methode** liefert nicht immer einen eindeutigen Knick. Zeigt die Kurve mehrere Knicke, testet man die zugehörigen k einzeln.
- **t-SNE** dient nur der Visualisierung. Die Abstände zwischen den Clustern im Plot sind nicht quantitativ interpretierbar.
- Das Clustering geschieht auf den **skalierten** Originaldaten, nicht auf der t-SNE-Projektion.